# Urban Mobility Analytics MVP
## Notebook 04: Integrated MVP Insights & Policy Implications

This notebook integrates the demand patterns (SUBE) and network supply (GTFS) with administrative boundaries to generate actionable public policy insights.

### Objectives:
1. **Overlay demand vs. supply** at the administrative scale (Comunas).
2. **Calculate a proxy index** for public transit accessibility.
3. **Discuss transit efficiency and policy design** enabled by this dataset.

---

### 1. Data Ingestion & Integration

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import yaml

with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Load processed tables
metrics_df = pd.read_parquet(os.path.join('..', config['sube']['mobility_metrics_output']))
stops_df = pd.read_parquet(os.path.join('..', config['gtfs']['stops_output']))
comunas_gdf = gpd.read_file(os.path.join('..', config['paths']['comunas_shapefile']))

print("Datasets loaded successfully.")

### 2. Supply vs. Demand Overlay Analysis

In [ ]:
# Calculating overall demand averages
avg_daily_transactions = metrics_df['total_transacciones'].mean()
avg_active_cards = metrics_df['total_tarjetas_activas'].mean()

print(f"Average Daily Transactions in system: {avg_daily_transactions:,.2f}")
print(f"Average Daily Active Cards: {avg_active_cards:,.2f}")

### 3. Public Transport Density Indicators

In [ ]:
# Convert stops to a GeoDataFrame to map their spatial density
stops_gdf = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df['stop_lon'], stops_df['stop_lat']),
    crs="EPSG:4326"
)

if comunas_gdf.crs != "EPSG:4326":
    comunas_gdf = comunas_gdf.to_crs("EPSG:4326")

# Spatial Join
joined = gpd.sjoin(stops_gdf, comunas_gdf, how="inner", predicate="within")
commune_col = [c for c in comunas_gdf.columns if 'COMUNA' in c.upper()][0]
stops_per_commune = joined.groupby(commune_col).size().reset_index(name='stops_count')

# Calculate area in km² (using equal-area projection for measurement)
comunas_projected = comunas_gdf.to_crs(epsg=3003) # Local meters-based projection
comunas_gdf['area_km2'] = comunas_projected.geometry.area / 1e6

comunas_gdf = comunas_gdf.merge(stops_per_commune, on=commune_col, how='left').fillna(0)
comunas_gdf['stops_density'] = comunas_gdf['stops_count'] / comunas_gdf['area_km2']

comunas_gdf[[commune_col, 'stops_count', 'area_km2', 'stops_density']].head()

### 4. Policy Insights & Low-Carbon Mobility Indicators

The integration of transit service supply data with aggregate public transport demand provides key insights for carbon emissions reduction and network efficiency:

* **Baseline Modal Split:** Visualizing the transaction proportions by travel mode (Colectivo, Subte, Tren) sets a baseline split.
* **Supply-Demand Matching:** Communes with high density of active cards but low transit stop density are priority regions for network optimization. Matching service schedules to demand prevents empty bus runs (reducing diesel emissions).
* **Equity and Gender-Perspectives:** Integrating spatial access indicators (off-peak trip analysis, travel time distributions) aids in making public transport safer and more accessible, directly encouraging modal shift from private vehicles (high carbon) to public transport (low carbon).